This code does the following:

- loads the idtracker trajectories, processes them to extract features 
- downsamples and smooths the data
- fits Gaussian HMM with desired number of states

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib
from hmmlearn import hmm
import pickle
import os


Load and preporces trajectories, extract features

In [ ]:
df = pd.read_csv("file path to idtracker trajectories.csv")

fps = 60

dx1, dy1 = np.gradient(df["x1"]), np.gradient(df["y1"])
dx2, dy2 = np.gradient(df["x2"]), np.gradient(df["y2"])

# Speed and acceleration
speed1 = np.sqrt(dx1**2 + dy1**2)
speed2 = np.sqrt(dx2**2 + dy2**2)
acc1 = np.gradient(speed1)
acc2 = np.gradient(speed2)

# Heading and angular velocity
heading1 = np.arctan2(dy1, dx1)
heading2 = np.arctan2(dy2, dx2)
angvel1 = np.gradient(heading1)
angvel2 = np.gradient(heading2)

# Inter-fish distance and relative motion
dist = np.sqrt((df["x1"] - df["x2"])**2 + (df["y1"] - df["y2"])**2)
rel_speed = np.abs(speed1 - speed2)
rel_acc = np.abs(acc1 - acc2)
heading_diff = np.angle(np.exp(1j * (heading1 - heading2))) 
heading_align = np.cos(heading_diff)  # 1 = aligned, -1 = opposite

# Construct feature matrix
X = np.column_stack([
    speed1, speed2,
    acc1, acc2,
    angvel1, angvel2,
    dist, rel_speed, rel_acc,
    heading_diff, heading_align
])
X = np.nan_to_num(X)

feature_names = [
    "speed1", "speed2",
    "acc1", "acc2",
    "angvel1", "angvel2",
    "dist", "rel_speed", "rel_acc",
    "heading_diff", "heading_align"
]
print("Feature matrix shape:", X.shape)


Downsample and smooth the features

In [ ]:
window = fps  # 1-second rolling mean
X_df = pd.DataFrame(X)
X_down = X_df.rolling(window, center=True).mean().iloc[::window, :].dropna().reset_index(drop=True)
print("Downsampled shape:", X_down.shape)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_down)

# optional smoothing after scaling (over a 3 second window)
X_smooth = pd.DataFrame(X_scaled).rolling(3, center=True).mean().bfill().ffill().values
print("Smoothed feature matrix:", X_smooth.shape)

# Save scaler if required
scaler_path = "desired file path for saving scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

Fit Gaussian HMM

In [ ]:
n_states = 5 #adjust number of states

model = hmm.GaussianHMM(
    n_components=n_states,
    covariance_type='tied',  #can be adjusted if required
    n_iter=500,
    tol=1e-3,
    min_covar=1e-3,
    random_state=42,
    verbose=True
)

model.fit(X_smooth)
labels = model.predict(X_smooth)
print("Converged:", model.monitor_.converged)

print("Transition matrix:\n", np.round(model.transmat_, 2))

# save model if necessary
save_path = "desired file path for saving hmm_model.pkl"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved at: {save_path}")